# CTCL (MF) atlas — Visium spatial download & ingest

First step of the spatial arm. Fetches the **Li/Strobl/Poyner 2024** 10x Visium sections,
reads them with `scanpy.read_visium`, concatenates into one AnnData, and writes a clean,
**resolVI-ready** object (`data/Li2024_atlas/visium/ctcl_visium.h5ad`).

Data provenance (see `SPATIAL_data_map.md`): there is **no processed Visium h5ad** published;
both accessions host per-section **spaceranger output tarballs** on EMBL-EBI BioStudies:
- **E-MTAB-13614** — 8 CTCL sections `CTCL1..8_spaceranger_output.tar`
- **E-MTAB-14559** — 15 healthy sections `WS_D_SKNsp<id>.tar.gz`

> ⚠️ Heavy steps (download ~8.5 GB, extract, full read) run on the **GPU/compute kernel**, not the
> login node. `sc.read_visium` per section is light; concatenation of 23 sections is the memory
> step. resolVI training is a **later** notebook.

In [ ]:
from pathlib import Path
import json, tarfile, subprocess, urllib.request
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

np.random.seed(0)


def _resolve_nb_dir():
    start = Path(__file__).parent.resolve() if "__file__" in globals() else Path.cwd()
    for base in [start, *start.parents]:
        for sub in [Path("."), Path("MF")]:
            cand = base / sub
            if cand.name == "MF" and (cand / "data").exists():
                return cand.resolve()
    raise FileNotFoundError(f"could not locate MF/data starting from {start}")

NB_DIR   = _resolve_nb_dir()
DATA_DIR = NB_DIR / "data"
FIG_DIR  = NB_DIR / "figures"; FIG_DIR.mkdir(exist_ok=True)
VISIUM_DIR = DATA_DIR / "Li2024_atlas" / "visium"; VISIUM_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR    = VISIUM_DIR / "raw";     RAW_DIR.mkdir(exist_ok=True)     # downloaded tarballs
EXTRACT_DIR= VISIUM_DIR / "sections"; EXTRACT_DIR.mkdir(exist_ok=True) # untarred spaceranger outs
VISIUM_H5AD = VISIUM_DIR / "ctcl_visium.h5ad"                          # output

sc.settings.set_figure_params(dpi=80, facecolor="white", figsize=(5, 3))
sc.settings.figdir = str(FIG_DIR)

# BioStudies accessions (verified 2026-07-12; no processed h5ad exists)
ACC_CTCL    = "E-MTAB-13614"   # 8 CTCL Visium sections (ENA ERP155847)
ACC_HEALTHY = "E-MTAB-14559"   # 15 healthy Visium sections (ENA ERP165340)
FILES_URL    = "https://www.ebi.ac.uk/biostudies/files/{acc}/{fname}"          # downloads
FILELIST_URL = "https://www.ebi.ac.uk/biostudies/api/v1/files/{acc}?length=1000"  # manifest

print("scanpy", sc.__version__)
print("NB_DIR     ", NB_DIR)
print("VISIUM_DIR ", VISIUM_DIR)

## 1. Resolve the file manifest

Query the BioStudies API for each accession's file list (authoritative). We keep only the
per-section spaceranger tarballs (`.tar` / `.tar.gz`), dropping the `.sdrf`/`.idf` metadata files
(fetched separately below).

In [ ]:
def fetch_manifest(acc):
    with urllib.request.urlopen(FILELIST_URL.format(acc=acc), timeout=60) as r:
        d = json.load(r)
    return [(f["path"], int(f.get("Size", 0) or 0)) for f in d["data"]]

def section_tarballs(acc):
    return [(p, sz) for p, sz in fetch_manifest(acc)
            if p.endswith(".tar") or p.endswith(".tar.gz")]

ctcl_files    = section_tarballs(ACC_CTCL)
healthy_files = section_tarballs(ACC_HEALTHY)

def _section_id(fname):
    # CTCL3_spaceranger_output.tar -> CTCL3 ; WS_D_SKNsp10264994.tar.gz -> WS_D_SKNsp10264994
    return fname.split("_spaceranger")[0].replace(".tar.gz", "").replace(".tar", "")

MANIFEST = (
    [dict(acc=ACC_CTCL, fname=p, size=sz, section=_section_id(p), condition="CTCL")    for p, sz in ctcl_files] +
    [dict(acc=ACC_HEALTHY, fname=p, size=sz, section=_section_id(p), condition="healthy") for p, sz in healthy_files]
)
manifest = pd.DataFrame(MANIFEST)
manifest["size_MB"] = (manifest["size"] / 1e6).round(0).astype(int)
print(f"{len(ctcl_files)} CTCL + {len(healthy_files)} healthy = {len(manifest)} sections, "
      f"{manifest['size_MB'].sum()/1000:.1f} GB total")
manifest[["condition", "section", "fname", "size_MB"]]

## 2. Download the tarballs

`wget -c` (resume-safe) into `raw/`. **~8.5 GB across 23 files** — run on the compute node.
Set `FETCH` to control scope: start with CTCL only (`"CTCL"`, ~2.4 GB), then `"all"` once verified.
Already-complete files are skipped.

In [ ]:
FETCH = "all"   # "CTCL" | "healthy" | "all"

def _select(df, which):
    if which == "all": return df
    return df[df["condition"] == ("CTCL" if which == "CTCL" else "healthy")]

def download(row):
    url = FILES_URL.format(acc=row.acc, fname=row.fname)
    dest = RAW_DIR / row.fname
    if dest.exists() and dest.stat().st_size >= row.size:
        print(f"  skip (have) {row.fname}"); return dest
    print(f"  wget {row.fname} ({row.size_MB} MB)")
    subprocess.run(["wget", "-q", "-c", "-O", str(dest), url], check=True)
    return dest

todo = _select(manifest, FETCH)
print(f"Downloading {len(todo)} files ({todo['size_MB'].sum()/1000:.1f} GB) -> {RAW_DIR}")
for row in todo.itertuples():
    download(row)
print("done")

### Section-level metadata (SDRF)

Small text files with per-section disease / sex / age. Parsed for annotation.

In [ ]:
def load_sdrf(acc):
    url = FILES_URL.format(acc=acc, fname=f"{acc}.sdrf.txt")
    df = pd.read_csv(url, sep="\t")
    keep = {"Source Name": "section"}
    for c in df.columns:
        cl = c.lower()
        if "characteristics[disease]" in cl: keep[c] = "disease"
        elif "characteristics[sex]" in cl:   keep[c] = "sex"
        elif "characteristics[age]" in cl:    keep[c] = "age"
    out = df[list(keep)].rename(columns=keep).drop_duplicates("section")
    return out

sdrf = pd.concat([load_sdrf(ACC_CTCL), load_sdrf(ACC_HEALTHY)], ignore_index=True)
sdrf = sdrf.set_index("section")
sdrf

## 3. Extract + read each section

Untar into `sections/<id>/`, locate the spaceranger `outs` dir (the folder containing `spatial/`),
and read with `sc.read_visium`. `library_id = section` so `uns['spatial']` keys stay unique for
plotting after concatenation.

In [ ]:
def extract(row):
    tar_path = RAW_DIR / row.fname
    out_dir = EXTRACT_DIR / row.section
    if not out_dir.exists():
        out_dir.mkdir(parents=True)
        with tarfile.open(tar_path, "r:*") as t:
            t.extractall(out_dir)
    return out_dir

def find_visium_dir(root):
    # spaceranger outs contains spatial/scalefactors_json.json + filtered_feature_bc_matrix(.h5)
    hits = list(root.rglob("spatial/scalefactors_json.json"))
    if not hits:
        raise FileNotFoundError(f"no spaceranger spatial/ under {root}")
    return hits[0].parent.parent  # the 'outs'-level dir

def read_section(row):
    vdir = find_visium_dir(extract(row))
    h5 = "filtered_feature_bc_matrix.h5"
    ad = sc.read_visium(vdir, count_file=h5 if (vdir / h5).exists() else None,
                        library_id=row.section)
    ad.var_names_make_unique()
    ad.obs["section"] = row.section
    ad.obs["condition"] = row.condition
    return ad

FETCHED = sorted({p.name for p in RAW_DIR.glob("*.tar*")})
sections = manifest[manifest["fname"].isin(FETCHED)]
print(f"reading {len(sections)} downloaded sections")
adatas = {row.section: read_section(row) for row in sections.itertuples()}
for sid, ad in adatas.items():
    print(f"  {sid:20s} {ad.n_obs:6d} spots x {ad.n_vars} genes")

## 4. Concatenate + annotate

Inner-join on genes (identical spaceranger reference), keep per-section `uns['spatial']` (images +
scalefactors) so `sc.pl.spatial` still works, and merge SDRF metadata.

In [ ]:
import anndata as ad_

merged_spatial = {}
for adx in adatas.values():
    merged_spatial.update(adx.uns.get("spatial", {}))

# section/condition already on each obs; concat a list and keep obsm/var aligned
adata = ad_.concat(list(adatas.values()), join="inner", index_unique="-", merge="unique")
adata.uns["spatial"] = merged_spatial

# section-level metadata
meta = sdrf.reindex(adata.obs["section"].values)
for c in meta.columns:
    adata.obs[c] = meta[c].values
adata.obs["section"] = adata.obs["section"].astype("category")
adata.obs["condition"] = adata.obs["condition"].astype("category")

print(adata)
print("\nsections:"); print(adata.obs["section"].value_counts())

## 5. QC per section

Spots, median counts/genes, mito fraction per section; `sc.pl.spatial` of one CTCL section
(sanity of coords + image).

In [ ]:
adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True, percent_top=None)

qc = (adata.obs.groupby("section", observed=True)
      .agg(spots=("total_counts", "size"),
           median_counts=("total_counts", "median"),
           median_genes=("n_genes_by_counts", "median"),
           pct_mt=("pct_counts_mt", "median"))
      .round(1))
print(qc)

_ctcl = [s for s in adata.obs["section"].cat.categories if str(s).startswith("CTCL")]
if _ctcl:
    s0 = _ctcl[0]
    sub = adata[adata.obs["section"] == s0].copy()
    sc.pl.spatial(sub, library_id=s0, color="total_counts", size=1.4,
                  title=f"{s0}: total counts/spot", show=False)
    plt.tight_layout(); plt.savefig(FIG_DIR / f"visium_qc_{s0}.png", dpi=120); plt.show()

## 6. resolVI-ready object + write

`scvi.external.RESOLVI` needs, per section: **raw counts** in a layer, **`obsm['X_spatial']`**
(default `spatial_rep`), and a **section batch key** for per-slice neighbor graphs. read_visium
gives raw counts in `.X` and coords in `.obsm['spatial']`; we stash both explicitly.

⚠️ resolVI targets *single-cell-resolved* spatial data; Visium spots are multi-cell, so its
neighbor-diffusion denoising is a partial fit — documented, not resolved here.

In [ ]:
adata.layers["counts"] = adata.X.copy()          # raw counts (read_visium X is raw)
adata.obsm["X_spatial"] = adata.obsm["spatial"].astype(float)

assert adata.layers["counts"].max() == int(adata.layers["counts"].max()), "counts not integer"
assert adata.obsm["X_spatial"].shape[1] == 2
assert adata.obs["section"].dtype.name == "category"

adata.write_h5ad(VISIUM_H5AD)
print("wrote", VISIUM_H5AD)
print(adata)

## 7. resolVI setup dry-check (no training)

Confirm the object is resolVI-consumable: run `setup_anndata` on **one small section** so the
per-section neighbor graph builds and `index_neighbor` / `distance_neighbor` populate. **No
training here** — the real fit is a later GPU notebook.

In [ ]:
from scvi.external import RESOLVI

s0 = adata.obs["section"].cat.categories[0]
probe = adata[adata.obs["section"] == s0].copy()
RESOLVI.setup_anndata(probe, layer="counts", batch_key="section", prepare_data=True,
                      prepare_data_kwargs={"spatial_rep": "X_spatial"})
print("index_neighbor  ", probe.obsm["index_neighbor"].shape)
print("distance_neighbor", probe.obsm["distance_neighbor"].shape)
print("resolVI setup OK on", s0)

## Summary

- **Sections:** 8 CTCL (`E-MTAB-13614`) + 15 healthy (`E-MTAB-14559`) 10x Visium — spaceranger
  tarballs from BioStudies (no processed h5ad exists upstream).
- **Output:** `data/Li2024_atlas/visium/ctcl_visium.h5ad` — raw counts in `layers['counts']`,
  coords in `obsm['spatial']`/`obsm['X_spatial']`, per-section images in `uns['spatial']`,
  `section` / `condition` / disease-sex-age obs.
- **resolVI-ready:** `setup_anndata(layer='counts', batch_key='section',
  prepare_data_kwargs={'spatial_rep':'X_spatial'})` verified on one section.
- **Gaps:** no per-spot cell-type labels (`labels_key`) — Visium spots are multi-cell; labels
  would come from deconvolution later. resolVI resolution mismatch noted above.

**Next:** resolVI training notebook (GPU), and/or join to the scRNA atlas for niche analysis.